# Import the models


In [3]:
import chromadb
import numpy as np
import pandas as pd
import bm25s
from pytubefix import YouTube, AsyncYouTube
import re
from nltk.tokenize import sent_tokenize

from IPython.display import Markdown
from chromadb import Documents, EmbeddingFunction, Embeddings

resource module not available on Windows


c:\Users\saker\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

client = genai.Client()

# Video caption extraction

In [5]:
async def extract_text_from_video(video_url):
    yt = YouTube(video_url) 
    ayt = AsyncYouTube(video_url)
    title = await ayt.title()
    caption = yt.captions['a.en']
    text = caption.generate_srt_captions()
    pattern = r"\d+\s*\n(\d{2}:\d{2}:\d{2},\d{3}) --> .*?\n(.*?)(?:\n\n|\Z)"
    result = re.sub(pattern, formatter, text, flags=re.DOTALL)
    text_final = re.sub(r'\s+', ' ', result).strip()
    return text_final , title

def formatter(match):
    timestamp = match.group(1)
    # On remplace les sauts de ligne internes par des espaces et on nettoie les bords
    text = match.group(2).replace("\n", " ").strip()
    # Format de sortie : Texte (Timestamp)
    return f"{text} ({timestamp}) "

In [6]:
caption, title = await extract_text_from_video("https://www.youtube.com/watch?v=UabBYexBD4k")
print(f"Extracted caption text from video '{title}': {caption[:200]}...")

Extracted caption text from video 'Is RAG Still Needed? Choosing the Best Approach for LLMs': There's a fundamental truth about LLMs, (00:00:00,320) large language models. They are frozen (00:00:04,240) in time. They know everything about our (00:00:08,240) world up until their training cutoff...


In [7]:
pattern = r'^[a-zA-Z0-9][a-zA-Z0-9._-]{1,510}[a-zA-Z0-9]$'
caption_sentences = sent_tokenize(caption)
title_clean = re.sub(r'[^\w\s-]', '', title).strip().replace(' ', '_')
print(f"Cleaned title for collection name: '{title_clean}'")
print(f"Extracted {len(caption_sentences)} sentences from video captions.")
print(f"First 5 sentences from video captions:\n{caption_sentences[:5]}")

Cleaned title for collection name: 'Is_RAG_Still_Needed_Choosing_the_Best_Approach_for_LLMs'
Extracted 100 sentences from video captions.
First 5 sentences from video captions:
["There's a fundamental truth about LLMs, (00:00:00,320) large language models.", 'They are frozen (00:00:04,240) in time.', 'They know everything about our (00:00:08,240) world up until their training cutoff (00:00:11,840) date and absolutely nothing about what (00:00:14,559) happened 5 minutes ago.', 'Nor do they know (00:00:16,800) anything about your private data, your (00:00:19,279) internal wiks, your proprietary (00:00:21,920) codebase.', 'And if we do want an LLM to (00:00:23,519) know any of that stuff, well, we have to (00:00:26,560) solve the problem of context injection.']


# Embedding Functions

In [8]:
import requests
class OpenrouterEmbeddingFunction(EmbeddingFunction):
  def __call__(self, input: Documents) -> Embeddings:
    response = requests.post(
      "https://openrouter.ai/api/v1/embeddings",
      headers={
        "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
        "Content-Type": "application/json",
      },
      json={
        "model": "nvidia/llama-nemotron-embed-vl-1b-v2:free",
        "input": input
      }
    )
    data = response.json()
    embedding = data["data"][0]["embedding"]
    return embedding

# Chroma database Creation

In [11]:
def create_chroma_db(path, documents, name):
  chroma_client = chromadb.PersistentClient(path=path)
  db = chroma_client.create_collection(
      name=name,
      embedding_function=OpenrouterEmbeddingFunction()
  )

  for i, d in enumerate(documents):
    db.add(
      documents=d,
      ids=str(i)
    )
  return db

db_path = r"C:\Users\saker\Desktop\RAG\simple\google_courses\db\chroma_db"
# collection_name = title_clean
collection_name = "ss45ff"
db = create_chroma_db(db_path, caption_sentences, collection_name)

C:\Users\saker\AppData\Local\Temp\ipykernel_6452\2447274403.py:5: DeprecationWarning: The class OpenrouterEmbeddingFunction does not implement __init__. This will be required in a future version.
  embedding_function=OpenrouterEmbeddingFunction()


In [12]:
sample_data = db.get(include=['documents', 'embeddings'])

df = pd.DataFrame({
    "IDs": sample_data['ids'][:3],
    "Documents": sample_data['documents'][:3],
    "Embeddings": [str(emb)[:50] + "..." for emb in sample_data['embeddings'][:3]]  # Truncate embeddings
})

df

,IDs,Documents,Embeddings
0,0,"There's a fundamental truth about LLMs, (00:00...",[ 0.00712204 0.02151489 -0.00277138 ... 0.03...
1,1,"They are frozen (00:00:04,240) in time.",[-0.02052307 0.00940704 0.00807953 ... -0.03...
2,2,"They know everything about our (00:00:08,240) ...",[-0.00450516 0.00865936 0.00432968 ... -0.01...


In [13]:
def get_relevant_passage(query, db):
  passage = db.query(query_texts=[query], n_results=30)['documents'][0]
  return passage

In [14]:
passage = get_relevant_passage("why RAG is dead ?", db)
passage_text = "\n".join([f"- {p}" for p in passage])
print(passage_text)

- So, is rag dead?
- A production rag system.
- Is rag becoming (00:03:49,920) an unnecessary complexity layer?
- You basically had to use rag.
- (00:07:03,520) And because rag only shows the model a (00:07:05,520) few isolated snapshots, the model never (00:07:08,080) sees the full picture required to spot (00:07:10,960) the missing pieces.
- But with rag, (00:09:24,800) we're giving the model less noise.
- (00:07:35,039) Well, not quite because while long (00:07:37,840) context wins on simplicity, rag still (00:07:40,479) has a place.
- It's rag retrieval augmented (00:00:46,879) generation.
- A rag (00:06:13,120) is fundamentally designed to retrieve (00:06:15,600) what exists.
- Now, rag also (00:08:20,160) has to process that manual, but it only (00:08:23,360) pays that processing cost once at (00:08:25,520) indexing time.
- Now, rag introduces a critical (00:05:16,880) point of failure here, the retrieval (00:05:20,479) step itself, because when a user asks a (00:05:22,880) quest

# Application

In [15]:
def make_prompt(query, relevant_passage):
  escaped = relevant_passage.replace("'", "").replace('"', "").replace("\n", " ")
  prompt = ("""
    Tu es un assistant expert en analyse de contenu vidéo. 
    Ta tâche est de répondre à la question de l'utilisateur en utilisant UNIQUEMENT les extraits de sous-titres fournis ci-dessous.

    RÈGLES IMPORTANTES :
    1. Si la réponse ne se trouve pas dans les extraits, dis : "Je n'ai pas trouvé cette information dans les sous-titres de la vidéo."
    2. Ne pas inventer d'informations (pas d'hallucinations).
    3. Les extraits peuvent être fragmentés, reconstruis le sens logique si nécessaire.
    4. Cite les timestamps si ils sont disponibles dans le contexte.

    ---
    CONTEXTE (Extraits de la vidéo) :
    {relevant_passage}
    ---

    QUESTION DE L'UTILISATEUR :
    {query}

    RÉPONSE :
  """).format(query=query, relevant_passage=escaped)

  return prompt

In [16]:
query = "what's the video about ?"
passage = get_relevant_passage(query, db)
#transform the passage list into a text context add - before each sentence to make it more clear for the model
passage_text = "\n".join([f"- {p}" for p in passage])
prompt = make_prompt(query, passage_text)
Markdown(prompt)


    Tu es un assistant expert en analyse de contenu vidéo. 
    Ta tâche est de répondre à la question de l'utilisateur en utilisant UNIQUEMENT les extraits de sous-titres fournis ci-dessous.

    RÈGLES IMPORTANTES :
    1. Si la réponse ne se trouve pas dans les extraits, dis : "Je n'ai pas trouvé cette information dans les sous-titres de la vidéo."
    2. Ne pas inventer d'informations (pas d'hallucinations).
    3. Les extraits peuvent être fragmentés, reconstruis le sens logique si nécessaire.
    4. Cite les timestamps si ils sont disponibles dans le contexte.

    ---
    CONTEXTE (Extraits de la vidéo) :
    - Now this (00:02:03,119) works but it does rely on something. - (00:10:57,519) - The model gets to see everything. - So where does this leave (00:10:22,079) us? - They are frozen (00:00:04,240) in time. - Let me know in (00:10:54,320) the comments. - But how about (00:10:48,399) you? - And we (00:05:48,880) actually have a name for this. - Well, thats something like 250k (00:08:05,680) of tokens. - (00:04:20,320) Well, it is quite heavy. - You decide. - You need a a vector database to (00:04:36,160) store it. - Its basically a lot (00:04:44,880) of moving parts, a lot of places for (00:04:47,199) things to break. - Is the (00:07:29,199) vector database destined for the museum (00:07:32,560) of things we needed in 2024? - We break them (00:01:12,240) up into smaller chunks and we pass them (00:01:15,280) through to an embedding model and the (00:01:19,920) embedding model takes those chunks and (00:01:24,159) it turns them into vectors and those (00:01:27,040) vectors are then stored in a dedicated (00:01:31,360) vector database. - And it tries to (00:05:36,000) find the closest match. - And vectors are (00:05:30,720) basically just like a really long series (00:05:32,160) of numbers in an array. - But what if the answer lies in (00:06:24,800) whats not in the database? - It retrieves snippets from (00:06:56,160) the requirements doc. - Now this is really the model (00:02:25,280) native solution because you skip the (00:02:28,160) database here and you skip the embedding (00:02:30,560) model. - Youre going to (00:04:30,880) need a embedding model to encode the (00:04:33,040) data. - Now ahead of time we take some of (00:00:59,199) the documents that we want to give to (00:01:02,800) this LLM. - So, is rag dead? - It forces the model to (00:09:41,279) focus on the signal and not the noise. - And we need to do that every (00:08:10,400) time we make a user query and we put (00:08:13,919) this document in the prompt. - They know everything about our (00:00:08,240) world up until their training cutoff (00:00:11,840) date and absolutely nothing about what (00:00:14,559) happened 5 minutes ago. - Some of them have, you know, a (00:03:14,239) million tokens plus. - So thats reason number one. - But with rag, (00:09:24,800) were giving the model less noise. - So now the (00:01:49,360) context window has the user prompt, but (00:01:51,680) it also has all of these chunks that we (00:01:54,880) have taken from the vector database and (00:01:58,240) together (00:02:01,200) this forms the context window.
    ---

    QUESTION DE L'UTILISATEUR :
    what's the video about ?

    RÉPONSE :
  

In [17]:
MODEL_ID = "gemini-2.5-flash"
answer = client.models.generate_content(
    model = MODEL_ID,
    contents = prompt
)
Markdown(answer.text)

La vidéo parle de la manière dont les documents sont décomposés en "plus petits morceaux" et passés à un "modèle d'embedding" qui les transforme en "vecteurs" stockés dans une "base de données vectorielle dédiée" (00:01:12,240 - 00:01:31,360). Ces "morceaux" de la base de données vectorielle, combinés avec la requête de l'utilisateur, forment la "fenêtre de contexte" (00:01:49,360 - 00:02:01,200).

La vidéo pose également la question "So, is rag dead?", décrivant le RAG (Retrieval Augmented Generation) comme un moyen de "donner moins de bruit au modèle" (00:09:24,800) et de le forcer "à se concentrer sur le signal et non sur le bruit" (00:09:41,279). Elle explore si la "base de données vectorielle est destinée au musée" (00:07:29,199 - 00:07:32,560) et mentionne une "solution native au modèle" qui permet de "sauter la base de données" et "le modèle d'embedding" (00:02:25,280 - 00:02:30,560).